# The Jacobi Method
The Jacobi method updates every unknown using values from the previous iteration. Because the new components do not depend on one another, their updates can be computed in parallel.

> __Learning Objectives:__
>
> By the end of this algorithm notebook, you should be able to:
> - **Derive the update:** Use the diagonal part of the system matrix to convert a residual into a correction.
> - **Follow a Jacobi sweep:** Compute every new component from the same previous iterate and explain how this differs from Gauss–Seidel.
> - **Assess convergence and stopping:** Identify the iteration matrix and distinguish meeting the residual tolerance from reaching a correction limit.

This notebook specializes the [general iterative method](CHEME-5800-L6c-Lecture-GeneralIterativeMethod-Fall-2026.ipynb). The [companion example](CHEME-5800-L6c-Example-FunWithIterativeSolvers-Fall-2026.ipynb) compares its numerical results and computational cost with other solvers.

___

## Matrix splitting and independent updates
Consider $\mathbf{A}\mathbf{x}=\mathbf{b}$, where $\mathbf{A}\in\mathbb{R}^{n\times n}$ is nonsingular, $\mathbf{x}\in\mathbb{R}^{n}$ contains the $n$ unknowns, and $\mathbf{b}\in\mathbb{R}^{n}$ is the right-hand side. Assume every diagonal entry $a_{ii}$ is nonzero. Let $\mathbf{D}$, $\mathbf{L}$, and $\mathbf{U}$ contain the diagonal, strictly lower triangular, and strictly upper triangular entries of $\mathbf{A}$. The Jacobi splitting is:

$$
\mathbf{A}
=\underbrace{\mathbf{D}}_{\mathbf{M}}
+\underbrace{(\mathbf{L}+\mathbf{U})}_{-\mathbf{N}},
\qquad \mathbf{N}=-(\mathbf{L}+\mathbf{U}).
$$

At iteration $k$, define the residual by $\mathbf{r}^{(k)}=\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)}$. The correction $\mathbf{d}^{(k)}$ solves a diagonal system, so each component requires only division by its diagonal entry:

$$
\begin{aligned}
\mathbf{D}\mathbf{d}^{(k)}&=\mathbf{r}^{(k)},
&&d_i^{(k)}=\frac{r_i^{(k)}}{a_{ii}},\quad i=1,\ldots,n,\\
\mathbf{x}^{(k+1)}&=\mathbf{x}^{(k)}+\mathbf{d}^{(k)},
&&\text{apply the correction}.
\end{aligned}
$$

We do not need to construct an inverse matrix. Substituting the residual and collecting terms gives the equivalent update:

$$
\mathbf{D}\mathbf{x}^{(k+1)}
=\mathbf{b}-(\mathbf{L}+\mathbf{U})\mathbf{x}^{(k)}.
$$

Reading row $i$ gives the component update:

$$
x_i^{(k+1)}
=\frac{1}{a_{ii}}\left(b_i-\sum_{j\ne i}a_{ij}x_j^{(k)}\right),
\qquad i=1,\ldots,n.
$$

The sum uses only values from the previous iterate; an empty sum is zero. In a component-by-component implementation, keep that vector unchanged until all new components have been computed. Reusing newly computed values during the sweep would instead give the [Gauss–Seidel update](CHEME-5800-L6c-Algorithm-GaussSeidel-Fall-2026.ipynb).

___

## Algorithm and convergence
__Initialize__: Given the system matrix $\mathbf{A}\in\mathbb{R}^{n\times n}$ and right-hand side $\mathbf{b}\in\mathbb{R}^{n}$, choose an initial guess $\mathbf{x}^{(0)}\in\mathbb{R}^{n}$, an absolute residual tolerance $\epsilon>0$, and a nonnegative integer correction limit $\texttt{maxiter}$. Set $\texttt{converged}\gets\texttt{false}$ and the correction counter $k\gets0$.

While not $\texttt{converged}$ __do__:

1. Calculate the residual vector $\mathbf{r}^{(k)}\gets\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)}$ before changing the iterate.
2. Check for convergence:
   - If $\|\mathbf{r}^{(k)}\|_2<\epsilon$, __then__: set $\texttt{converged}\gets\texttt{true}$ and return $\mathbf{x}^{(k)}$ with this status.
   - Otherwise, if $k\ge\texttt{maxiter}$, __then__: print a __warning__ that the residual tolerance was not met and return $\mathbf{x}^{(k)}$ with $\texttt{converged}=\texttt{false}$.
3. Calculate the update direction: $d_i^{(k)}\gets r_i^{(k)}/a_{ii}$ for each $i=1,\ldots,n$.
4. Update the solution vector: $\mathbf{x}^{(k+1)}\gets\mathbf{x}^{(k)}+\mathbf{d}^{(k)}$.
5. Increment the correction counter: $k\gets k+1$.

Check the residual before the correction limit: an acceptable initial guess needs no update, and an approximation that first meets tolerance after the last allowed correction is still successful. Either return stops the algorithm immediately. A small residual means the equations are nearly satisfied; its relationship to solution error also depends on the conditioning of $\mathbf{A}$.

To assess convergence of the underlying iteration, write the update in stationary form:

$$
\mathbf{x}^{(k+1)}
=\underbrace{-\mathbf{D}^{-1}(\mathbf{L}+\mathbf{U})}_{\mathbf{G}_{J}}
\mathbf{x}^{(k)}
+\underbrace{\mathbf{D}^{-1}\mathbf{b}}_{\mathbf{c}}.
$$

Here $\mathbf{G}_{J}$ is the iteration matrix and $\mathbf{c}$ is the constant vector. The iteration converges to the solution from every initial guess if and only if:

$$
\rho(\mathbf{G}_{J})=\max_i|\lambda_i|<1,
$$

where $\lambda_i$ are its eigenvalues and $\rho$ is the spectral radius. Strict row diagonal dominance is one sufficient condition. The [lecture's convergence section](CHEME-5800-L6c-Lecture-GeneralIterativeMethod-Fall-2026.ipynb) derives this result using the absolute row sums of the Jacobi iteration matrix.

___

## Summary
We specialized the residual-correction method to a diagonal solve.

> __Key Takeaways:__
>
> - **Diagonal corrections:** We derived the Jacobi correction by dividing each residual component by the corresponding diagonal entry. This requires nonzero diagonal entries but no explicit inverse matrix.
> - **Independent updates:** We showed that every new component uses the same previous iterate. Keeping those values unchanged throughout the sweep preserves the Jacobi method and allows the component updates to run in parallel.
> - **Convergence and stopping:** We identified the iteration matrix and its convergence condition. Our pseudocode returns immediately when the residual meets tolerance or the correction limit is reached, reporting these outcomes separately.

Use the companion example to check the final residual before comparing solver timings.

___